# CNN raw I/Q — Tuni2025 GPS spoofing

Czysta wersja notebooka: bez używania `C7` jako niezależnego clean, z czasowym podziałem `C5` i testem held-out na `SS33`.

In [ ]:
from pathlib import Path
import random
import re
import gc

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_DIR = Path("/workspace/gps-spoofing")
RAW_DIR = PROJECT_DIR / "raw"
MODELS_DIR = PROJECT_DIR / "models"
REPORTS_DIR = PROJECT_DIR / "reports"
FIGURES_DIR = PROJECT_DIR / "figures"

for d in [MODELS_DIR, REPORTS_DIR, FIGURES_DIR]:
    d.mkdir(exist_ok=True)

SAMPLE_RATE_HZ = 50_000_000
WINDOW_SAMPLES = 131_072
MAX_WINDOWS_PER_FILE = 2_000
BATCH_SIZE = 256
NUM_WORKERS = 0
EPOCHS = 10
PATIENCE = 3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Scenariusze i etykiety

`C7` jest wykluczony, ponieważ okazał się kopią `C5`. `C5` dzielimy czasowo na train/val, a `SS33` zostaje testem held-out z etykietami czasowymi.

In [ ]:
def scenario_key(name: str) -> str:
    upper = name.upper()
    m = re.search(r"(SS|C)[-_ ]?(\d+)", upper)
    if m:
        return f"{m.group(1)}{int(m.group(2))}"
    return upper


def label_from_name(name: str) -> int:
    key = scenario_key(name)
    return 0 if key.startswith("C") or "clean" in name.lower() else 1


ATTACK_INTERVALS_S = {
    "SS33": [(70.0, 150.0)],
}

IGNORE_INTERVALS_S = {
    "SS33": [(65.0, 70.0), (150.0, 152.0)],
}

CLEAN_TIME_SPLITS_S = {
    "C5": {
        "train": [(0.0, 90.0)],
        "val": [(105.0, 150.0)],
    }
}

EXCLUDED_SCENARIOS = {"C7"}
UNKNOWN_SPOOF_POLICY = "file_level"


def overlaps(a_start: float, a_end: float, b_start: float, b_end: float) -> bool:
    return (a_end > b_start) and (a_start < b_end)


def window_overlaps_any_interval(window_start_s: float, window_end_s: float, intervals) -> bool:
    return any(overlaps(window_start_s, window_end_s, s, e) for s, e in intervals)


def label_window_from_time(file_name: str, window_start_s: float, window_end_s: float):
    key = scenario_key(file_name)
    file_label = label_from_name(file_name)

    if file_label == 0:
        return 0

    for s, e in IGNORE_INTERVALS_S.get(key, []):
        if overlaps(window_start_s, window_end_s, s, e):
            return None

    if key in ATTACK_INTERVALS_S:
        return int(window_overlaps_any_interval(window_start_s, window_end_s, ATTACK_INTERVALS_S[key]))

    if UNKNOWN_SPOOF_POLICY == "file_level":
        return 1
    if UNKNOWN_SPOOF_POLICY == "skip":
        return None

    raise ValueError(f"Unknown spoof policy: {UNKNOWN_SPOOF_POLICY}")


raw_files = sorted(RAW_DIR.glob("*.bin"))
completed_files = [p for p in raw_files if not Path(str(p) + ".aria2").exists()]
usable_files = [p for p in completed_files if scenario_key(p.name) not in EXCLUDED_SCENARIOS]

train_scenarios = ["C5", "SS17", "SS18", "SS20", "SS27", "SS28"]
val_scenarios = ["C5", "SS29"]
test_scenarios = ["SS33"]


def files_for_scenarios(files, scenarios):
    scenarios = set(scenarios)
    return [p for p in files if scenario_key(p.name) in scenarios]


train_files = files_for_scenarios(usable_files, train_scenarios)
val_files = files_for_scenarios(usable_files, val_scenarios)
test_files = files_for_scenarios(usable_files, test_scenarios)

assert train_files, "Brak plików treningowych."
assert val_files, "Brak plików walidacyjnych."
assert test_files, "Brak pliku testowego SS33."

files_df = pd.DataFrame([
    {
        "file": p.name,
        "scenario": scenario_key(p.name),
        "split": (
            "train" if p in train_files else
            "val" if p in val_files else
            "test" if p in test_files else
            "unused"
        ),
        "size_GB": p.stat().st_size / 1024**3,
        "file_label": label_from_name(p.name),
    }
    for p in completed_files
])

display(files_df.sort_values(["split", "scenario", "file"]))

## 2. Dataset raw I/Q

In [ ]:
class RawIQDatasetInt16(Dataset):
    def __init__(
        self,
        raw_files,
        window_samples=16_384,
        max_windows_per_file=1_000,
        normalize=True,
        per_window_standardize=True,
        sample_rate_hz=50_000_000,
        use_temporal_labels=True,
        time_split_name=None,
        shuffle_index=True,
    ):
        self.raw_files = list(raw_files)
        self.window_samples = int(window_samples)
        self.max_windows_per_file = int(max_windows_per_file)
        self.normalize = bool(normalize)
        self.per_window_standardize = bool(per_window_standardize)
        self.sample_rate_hz = float(sample_rate_hz)
        self.use_temporal_labels = bool(use_temporal_labels)
        self.time_split_name = time_split_name
        self.index = []
        self.skipped_windows = []
        self._memmaps = {}

        bytes_per_value = np.dtype(np.int16).itemsize

        for file_idx, p in enumerate(self.raw_files):
            file_size = p.stat().st_size
            value_count = file_size // bytes_per_value
            complex_count = value_count // 2
            total_windows = complex_count // self.window_samples
            if total_windows <= 0:
                continue

            key = scenario_key(p.name)
            split_cfg = CLEAN_TIME_SPLITS_S.get(key)

            if self.time_split_name is not None and split_cfg is not None:
                allowed_intervals = split_cfg.get(self.time_split_name, [])
                candidate_windows = []

                for start_s, end_s in allowed_intervals:
                    start_w = max(0, int(np.floor(start_s * self.sample_rate_hz / self.window_samples)))
                    end_w = min(total_windows - 1, int(np.ceil(end_s * self.sample_rate_hz / self.window_samples)) - 1)
                    if end_w >= start_w:
                        candidate_windows.append(np.arange(start_w, end_w + 1, dtype=np.int64))

                if not candidate_windows:
                    continue

                candidate_windows = np.unique(np.concatenate(candidate_windows))
                used_windows = min(len(candidate_windows), self.max_windows_per_file)
                selected_positions = np.linspace(0, len(candidate_windows) - 1, used_windows, dtype=np.int64)
                window_ids = candidate_windows[selected_positions]
            else:
                used_windows = min(total_windows, self.max_windows_per_file)
                window_ids = np.linspace(0, total_windows - 1, used_windows, dtype=np.int64)

            for w in window_ids:
                w = int(w)
                window_start_s = (w * self.window_samples) / self.sample_rate_hz
                window_end_s = ((w + 1) * self.window_samples) / self.sample_rate_hz

                if self.time_split_name is not None and split_cfg is not None:
                    allowed_intervals = split_cfg.get(self.time_split_name, [])
                    if not window_overlaps_any_interval(window_start_s, window_end_s, allowed_intervals):
                        self.skipped_windows.append((p.name, key, w, "outside_time_split"))
                        continue

                label = label_window_from_time(p.name, window_start_s, window_end_s) if self.use_temporal_labels else label_from_name(p.name)
                if label is None:
                    self.skipped_windows.append((p.name, key, w, "ignored_interval"))
                    continue

                self.index.append((file_idx, w, int(label), window_start_s, window_end_s, key))

        if shuffle_index:
            random.shuffle(self.index)

    def __len__(self):
        return len(self.index)

    def _get_memmap(self, file_idx):
        if file_idx not in self._memmaps:
            self._memmaps[file_idx] = np.memmap(self.raw_files[file_idx], dtype=np.int16, mode="r")
        return self._memmaps[file_idx]

    def __getitem__(self, idx):
        file_idx, window_idx, label, _, _, _ = self.index[idx]
        mm = self._get_memmap(file_idx)
        start = window_idx * self.window_samples * 2
        end = start + self.window_samples * 2
        iq = mm[start:end].reshape(-1, 2).astype(np.float32)

        if self.normalize:
            iq = iq / 32768.0

        if self.per_window_standardize:
            iq = iq - iq.mean(axis=0, keepdims=True)
            iq = iq / (iq.std(axis=0, keepdims=True) + 1e-6)

        x = torch.from_numpy(iq.T.copy())
        y = torch.tensor(label, dtype=torch.float32)
        return x, y


def dataset_index_df(ds: RawIQDatasetInt16) -> pd.DataFrame:
    rows = []
    for file_idx, window_idx, label, start_s, end_s, scen in ds.index:
        rows.append({
            "file": ds.raw_files[file_idx].name,
            "scenario": scen,
            "time_split_name": ds.time_split_name,
            "window_idx": window_idx,
            "window_start_s": start_s,
            "window_end_s": end_s,
            "label": int(label),
        })
    return pd.DataFrame(rows)

In [ ]:
train_ds = RawIQDatasetInt16(
    train_files,
    window_samples=WINDOW_SAMPLES,
    max_windows_per_file=MAX_WINDOWS_PER_FILE,
    normalize=True,
    per_window_standardize=True,
    sample_rate_hz=SAMPLE_RATE_HZ,
    use_temporal_labels=True,
    time_split_name="train",
)

val_ds = RawIQDatasetInt16(
    val_files,
    window_samples=WINDOW_SAMPLES,
    max_windows_per_file=MAX_WINDOWS_PER_FILE,
    normalize=True,
    per_window_standardize=True,
    sample_rate_hz=SAMPLE_RATE_HZ,
    use_temporal_labels=True,
    time_split_name="val",
)

test_ds = RawIQDatasetInt16(
    test_files,
    window_samples=WINDOW_SAMPLES,
    max_windows_per_file=MAX_WINDOWS_PER_FILE,
    normalize=True,
    per_window_standardize=True,
    sample_rate_hz=SAMPLE_RATE_HZ,
    use_temporal_labels=True,
    time_split_name=None,
)

train_index_df = dataset_index_df(train_ds)
val_index_df = dataset_index_df(val_ds)
test_index_df = dataset_index_df(test_ds)

print("Train windows:", len(train_ds), "skipped:", len(train_ds.skipped_windows))
print("Val windows:", len(val_ds), "skipped:", len(val_ds.skipped_windows))
print("Test windows:", len(test_ds), "skipped:", len(test_ds.skipped_windows))

for split_name, idx_df in [("train", train_index_df), ("val", val_index_df), ("test", test_index_df)]:
    print(f"\n{split_name} labels:")
    print(idx_df["label"].value_counts().sort_index())
    display(idx_df.groupby(["scenario", "label"]).size().unstack(fill_value=0))
    assert idx_df["label"].nunique() == 2, f"{split_name} ma tylko jedną klasę."
    assert "C7" not in set(idx_df["scenario"]), f"C7 pojawił się w {split_name}."

x0, y0 = train_ds[0]
print("Sample x:", x0.shape, "y:", y0.item(), "mean:", float(x0.mean()), "std:", float(x0.std()))

## 3. Model i trening

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

xb, yb = next(iter(train_loader))
print("Batch x:", xb.shape)
print("Batch y:", yb.shape)
print("Batch labels:", torch.unique(yb, return_counts=True))

In [ ]:
class BiggerIQCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Conv1d(128, 256, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


model = BiggerIQCNN().to(DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
def run_one_epoch(model, loader, optimizer=None, device=DEVICE):
    is_train = optimizer is not None
    model.train(is_train)
    loss_fn = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda" and is_train))

    total_loss = 0.0
    total = 0
    correct = 0
    all_y = []
    all_proba = []

    for x, y in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast("cuda", enabled=(device == "cuda")):
                logits = model(x)
                loss = loss_fn(logits, y)

            if is_train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        proba = torch.sigmoid(logits)
        pred = (proba >= 0.5).float()

        total_loss += loss.item() * x.size(0)
        correct += (pred == y).sum().item()
        total += y.numel()
        all_y.append(y.detach().cpu())
        all_proba.append(proba.detach().cpu())

    y_true = torch.cat(all_y).numpy()
    y_proba = torch.cat(all_proba).numpy()
    y_pred = (y_proba >= 0.5).astype(int)

    return {
        "loss": total_loss / total,
        "acc": correct / total,
        "y_true": y_true,
        "y_proba": y_proba,
        "y_pred": y_pred,
    }

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

history = []
best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")

    train_metrics = run_one_epoch(model, train_loader, optimizer=optimizer, device=DEVICE)
    val_metrics = run_one_epoch(model, val_loader, optimizer=None, device=DEVICE)

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
    }
    history.append(row)

    print(
        f"train_loss={row['train_loss']:.4f} train_acc={row['train_acc']:.4f} "
        f"val_loss={row['val_loss']:.4f} val_acc={row['val_acc']:.4f}"
    )

    if row["val_loss"] < best_val_loss:
        best_val_loss = row["val_loss"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

history_df = pd.DataFrame(history)

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(DEVICE)

history_df

## 4. Walidacja, threshold i test SS33

In [ ]:
def print_binary_metrics(y_true, y_proba, threshold=0.5, title="set"):
    y_pred = (y_proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print(f"\n{title} @ threshold={threshold:.3f}")
    print(cm)
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))
    print("FPR:", fp / (fp + tn) if (fp + tn) else np.nan)
    print("FNR:", fn / (fn + tp) if (fn + tp) else np.nan)
    print("Spoof recall:", tp / (tp + fn) if (tp + fn) else np.nan)
    print("Spoof precision:", tp / (tp + fp) if (tp + fp) else np.nan)
    print("ROC-AUC:", roc_auc_score(y_true, y_proba))
    print("PR-AUC:", average_precision_score(y_true, y_proba))
    return cm, y_pred


val_metrics = run_one_epoch(model, val_loader, optimizer=None, device=DEVICE)
val_y_true = np.array(val_metrics["y_true"])
val_y_proba = np.array(val_metrics["y_proba"])

print_binary_metrics(val_y_true, val_y_proba, threshold=0.5, title="Validation baseline")

In [ ]:
threshold_rows = []

for t in np.arange(0.05, 0.90, 0.01):
    y_pred_t = (val_y_proba >= t).astype(int)
    cm = confusion_matrix(val_y_true, y_pred_t, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    threshold_rows.append({
        "threshold": float(t),
        "accuracy": (tp + tn) / (tp + tn + fp + fn),
        "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
        "FNR": fn / (fn + tp) if (fn + tp) else np.nan,
        "spoof_recall": tp / (tp + fn) if (tp + fn) else np.nan,
        "spoof_precision": tp / (tp + fp) if (tp + fp) else np.nan,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    })

threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

candidates = threshold_df[threshold_df["FNR"] == 0.0].copy()
if len(candidates):
    best_row = candidates.sort_values(["FPR", "threshold"], ascending=[True, False]).iloc[0]
else:
    best_row = threshold_df.sort_values(["FNR", "FPR", "threshold"], ascending=[True, True, False]).iloc[0]

BEST_THRESHOLD = float(best_row["threshold"])
print("BEST_THRESHOLD:", BEST_THRESHOLD)
display(best_row.to_frame().T)

In [ ]:
test_metrics = run_one_epoch(model, test_loader, optimizer=None, device=DEVICE)
test_y_true = np.array(test_metrics["y_true"])
test_y_proba = np.array(test_metrics["y_proba"])
test_cm, test_y_pred = print_binary_metrics(
    test_y_true,
    test_y_proba,
    threshold=BEST_THRESHOLD,
    title="Test SS33",
)

## 5. Diagnostyka czasowa i wykresy

In [ ]:
def predict_dataset_with_metadata(model, ds, device, batch_size=256, threshold=0.5):
    model.eval()
    ordered_indices = sorted(range(len(ds.index)), key=lambda i: (ds.index[i][0], ds.index[i][1]))
    rows = []
    batch_x = []
    batch_meta = []

    def flush():
        nonlocal batch_x, batch_meta, rows
        if not batch_x:
            return
        x = torch.stack(batch_x).to(device, non_blocking=True)
        with torch.no_grad():
            proba = torch.sigmoid(model(x).view(-1)).detach().cpu().numpy()
        for meta, p in zip(batch_meta, proba):
            rows.append({**meta, "y_proba": float(p), "y_pred": int(p >= threshold)})
        batch_x, batch_meta = [], []

    for i in ordered_indices:
        file_idx, window_idx, label, start_s, end_s, scen = ds.index[i]
        x, _ = ds[i]
        batch_x.append(x)
        batch_meta.append({
            "file": ds.raw_files[file_idx].name,
            "file_idx": file_idx,
            "scenario": scen,
            "window_idx": window_idx,
            "window_start_s": start_s,
            "window_end_s": end_s,
            "y_true": int(label),
        })
        if len(batch_x) >= batch_size:
            flush()
    flush()
    return pd.DataFrame(rows)


test_diag_df = predict_dataset_with_metadata(
    model,
    test_ds,
    device=DEVICE,
    batch_size=BATCH_SIZE,
    threshold=BEST_THRESHOLD,
)

display(test_diag_df.head())
print_binary_metrics(
    test_diag_df["y_true"].to_numpy(),
    test_diag_df["y_proba"].to_numpy(),
    threshold=BEST_THRESHOLD,
    title="Temporal diagnostic SS33",
)

In [ ]:
import matplotlib.pyplot as plt

threshold = BEST_THRESHOLD
y_true = test_diag_df["y_true"].to_numpy()
y_proba = test_diag_df["y_proba"].to_numpy()
y_pred = (y_proba >= threshold).astype(int)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title(f"CNN confusion matrix, threshold={threshold:.2f}")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks([0, 1], ["clean", "spoof"])
plt.yticks([0, 1], ["clean", "spoof"])
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.colorbar(label="count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cnn_confusion_matrix.png", dpi=250)
plt.show()

fpr, tpr, _ = roc_curve(y_true, y_proba)
precision, recall, _ = precision_recall_curve(y_true, y_proba)
roc_auc = roc_auc_score(y_true, y_proba)
pr_auc = average_precision_score(y_true, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("CNN ROC curve")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cnn_roc_curve.png", dpi=250)
plt.show()

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("CNN Precision-Recall curve")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cnn_pr_curve.png", dpi=250)
plt.show()

In [ ]:
df_time = test_diag_df.sort_values("window_start_s").copy()
df_time["p_spoof_smooth"] = df_time["y_proba"].rolling(5, center=True, min_periods=1).mean()

plt.figure(figsize=(12, 4.8))
plt.scatter(df_time["window_start_s"], df_time["y_proba"], s=8, alpha=0.35, label="P(spoof) dla okna")
plt.plot(df_time["window_start_s"], df_time["p_spoof_smooth"], linewidth=2, label="średnia krocząca")
plt.axhline(threshold, linestyle="--", linewidth=1.5, label=f"threshold = {threshold:.2f}")
plt.axvline(70, linestyle="--", linewidth=1.5, label="start spoofingu ~70 s")
plt.axvline(150, linestyle="--", linewidth=1.5, label="koniec spoofingu ~150 s")
plt.xlabel("czas od początku nagrania SS33 [s]")
plt.ylabel("P(spoof)")
plt.title("CNN — detekcja delayed spoofingu w czasie dla SS33")
plt.ylim(-0.05, 1.05)
plt.grid(True)
plt.legend(loc="best")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "cnn_ss33_detection_over_time.png", dpi=250)
plt.show()

## 6. Zapis wyników

In [ ]:
EXPERIMENT_NAME = "cnn_raw_iq_int16_temporal_v6_clean_timesplit"

model_path = MODELS_DIR / f"{EXPERIMENT_NAME}.pt"
torch.save(model.state_dict(), model_path)

history_df.to_csv(REPORTS_DIR / f"{EXPERIMENT_NAME}_history.csv", index=False)
threshold_df.to_csv(REPORTS_DIR / f"{EXPERIMENT_NAME}_thresholds.csv", index=False)
train_index_df.to_csv(REPORTS_DIR / f"{EXPERIMENT_NAME}_train_index.csv", index=False)
val_index_df.to_csv(REPORTS_DIR / f"{EXPERIMENT_NAME}_val_index.csv", index=False)
test_index_df.to_csv(REPORTS_DIR / f"{EXPERIMENT_NAME}_test_index.csv", index=False)
test_diag_df.to_csv(REPORTS_DIR / f"{EXPERIMENT_NAME}_test_diagnostics.csv", index=False)

print("Saved model:", model_path)
print("Saved reports to:", REPORTS_DIR)
print("Saved figures to:", FIGURES_DIR)